# OrientedDet × SSDD (SAR ship finetune)

[SAR Ship Detection Dataset](https://github.com/TianwenZhang/Official-SSDD) (Zhang et al., 2021): ~1,160 chips, 2,456 ships, official last-digit train/test (~928 / ~232).

| | |
|---|---|
| Dataset | SSDD (RadarSat-2 / TerraSAR-X / Sentinel-1). **Download from the authors’ Google Drive** — this notebook does not vendor images. |
| Native format | `dataset.format: ssdd` (VOC XML, COCO, or DOTA layouts) |
| Recipe | [`configs/rotated_faster_rcnn/ssdd_le90_1x.json`](../configs/rotated_faster_rcnn/ssdd_le90_1x.json) — **12-epoch 1×** from `hf://rotated_faster_rcnn_dota_le90_1x`, keep-ratio **608**, pad-32 |
| HRSID | Larger SAR **benchmark** — not this notebook. See [Data guide — HRSID](../docs/user-guide/data.md#hrsid). |

Point `DATA_ROOT` at your unzipped dump (`/path/to/data/Official-SSDD-OPEN`). Chips are already small — **do not tile**. Polygons become OBBs only (not instance segmentation).

Expected AP50: **90.34%** held-out official test after the full 12-epoch recipe ([SSDD data guide](../docs/user-guide/data.md#ssdd-1x-faster-rcnn)). The 1-epoch smoke will not hit that number.


## 0. Install OrientedDet


In [ ]:
from pathlib import Path

def _find_repo() -> Path:
    here = Path.cwd().resolve()
    for cand in (here, *here.parents):
        if (cand / "pyproject.toml").is_file() and (cand / "oriented_det").is_dir():
            return cand
    return Path("..").resolve()

REPO = _find_repo()
if (REPO / "pyproject.toml").is_file() and (REPO / "oriented_det").is_dir():
    %pip install -q -e {REPO}
else:
    %pip install -q "git+https://github.com/DL4EO/oriented-det.git"

import oriented_det
from oriented_det.data import SSDDDataset

print("oriented_det", getattr(oriented_det, "__version__", "?"))
print("REPO", REPO)
assert hasattr(oriented_det.data, "SSDDDataset")


## 1. Paths & knobs

Official split: file numbers whose last digit is **1 or 9** are test. `val_split` defaults to that held-out test.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

DATA_ROOT = Path(os.environ.get("SSDD_DATA_ROOT", "/path/to/data/Official-SSDD-OPEN"))
WORK = Path(os.environ.get("SSDD_WORK", "./ssdd_tutorial_work")).resolve()
WORK.mkdir(parents=True, exist_ok=True)

MAX_TRAIN = 32
MAX_VAL = 16
NUM_EPOCHS = 1

print("DATA_ROOT", DATA_ROOT, "exists" if DATA_ROOT.is_dir() else "MISSING")
print("WORK", WORK)


## 2. Discover the dump


In [ ]:
from oriented_det.data.ssdd import SSDDDataset

train = SSDDDataset(DATA_ROOT, split="train", difficult_strategy="keep")
test = SSDDDataset(DATA_ROOT, split="test", difficult_strategy="keep")
print(f"train {len(train)}  test {len(test)}  classes {train.get_class_names()}")
sample = train[0]
print(sample.image_path.name, sample.width, sample.height, f"{len(sample.annotations)} ship(s)")
print("rbox0", sample.annotations[0].rbox if sample.annotations else None)


## 3. Optional: export DOTA folders

Native `format: ssdd` does **not** need this. Use it if you want `format: dota` loaders.


In [ ]:
# from oriented_det.data import export_ssdd_to_dota
# export_ssdd_to_dota(DATA_ROOT, WORK / "SSDD-dota", splits=("train", "test"))
print("Skip by default. Uncomment to write", WORK / "SSDD-dota")


## 4. Optional: 1-epoch Faster R-CNN smoke

Full 1× is **12 epochs** on all 928/232 chips (next section). This cell caps samples so it finishes quickly. Expect **low mAP** — not a benchmark.


In [ ]:
from oriented_det.train.config import TrainingExperimentConfig

recipe = REPO / "configs/rotated_faster_rcnn/ssdd_le90_1x.json"
cfg = TrainingExperimentConfig.load(recipe)
cfg.dataset.data_root = DATA_ROOT
cfg.dataset.max_train_samples = MAX_TRAIN
cfg.dataset.max_val_samples = MAX_VAL
cfg.training.num_epochs = NUM_EPOCHS
cfg.evaluation.compute_map_every_n_epochs = 1
smoke = WORK / "ssdd_smoke.json"
cfg.save(smoke)
print("wrote", smoke)
print("train with: odet train --config", smoke)


In [ ]:
import shutil
import subprocess

RUN_SMOKE = False  # set True on a GPU machine with SSDD unpacked
if not RUN_SMOKE:
    print("RUN_SMOKE=False — skip training. When True: odet train --config", smoke)
else:
    odet = shutil.which("odet")
    if not odet:
        raise RuntimeError("odet not on PATH — reinstall oriented-det")
    cmd = [odet, "train", "--config", str(smoke)]
    print(" ".join(cmd))
    subprocess.check_call(cmd)


## 5. Full 12-epoch 1× (benchmark)

Unmodified recipe: all 928 train / 232 test chips, 12 epochs, milestones 8/11, init `hf://rotated_faster_rcnn_dota_le90_1x`. `make eval-val` is **held-out official test**.

Expected AP50: **90.34%** held-out official test (`runs/rotated_faster_rcnn/20260918-130546`). Literature Faster R-CNN is often ~89%. Details: [SSDD 1× Faster R-CNN](../docs/user-guide/data.md#ssdd-1x-faster-rcnn). The 1-epoch smoke will not hit that number.

HRSID (5,604×800 chips) is documented in the [Data guide — HRSID](../docs/user-guide/data.md#hrsid), not here.


In [ ]:
import shutil
import subprocess

RUN_FULL = False  # set True for the 12-epoch recipe (all chips, GPU)
FULL_CFG = REPO / "configs/rotated_faster_rcnn/ssdd_le90_1x.json"

if not RUN_FULL:
    print("RUN_FULL=False — skip 12-epoch train.")
    print("When True, or from a shell:")
    print(f"  odet train --config {FULL_CFG}")
    print("Then held-out test:")
    print("  make eval-val EXPERIMENT=runs/rotated_faster_rcnn/<timestamp>")
else:
    odet = shutil.which("odet")
    if not odet:
        raise RuntimeError("odet not on PATH — reinstall oriented-det")
    cmd = [odet, "train", "--config", str(FULL_CFG)]
    print(" ".join(cmd))
    subprocess.check_call(cmd)
